# Parameter-Efficient Fine-Tuning (PEFT) with Adapters
## Sentiment Classification on SST-2 Dataset

Hey there. This notebook walks through a practical way to adapt large language models without breaking the bank on compute or storage. Instead of retweaking every single weight in BERT (all 110 million of them), we'll plug in lightweight "Adapters" that sit inside the transformer blocks. You train just these adapters, while the original model stays frozen. It's like upgrading a car's engine without replacing the entire chassis.

We're using SST-2, a solid benchmark for sentiment analysis, and we'll limit training to 10K examples to keep things speedy on a standard GPU. By the end, you'll see both the dramatic difference in parameter efficiency and how well this approach actually works.

**What we're building:**
1. Load and tokenize text the way BERT expects
2. Create a simple but effective Adapter block
3. Inject adapters into BERT's transformer layers
4. Train on SST-2 and run inference on custom sentences

## Installation & Imports

First, grab the libraries we need. This assumes you're on Colab or have a similar environment set up.

In [ ]:
import subprocess
import sys

packages = ['transformers', 'datasets', 'torch', 'numpy', 'matplotlib', 'tqdm']

for package in packages:
    try:
        __import__(package)
        print(f'✓ {package} already installed')
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f'✓ {package} installed')

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---\n
## Task 1: Preprocessing & BERT Tokenization\n
\n
BERT doesn't work with raw text. We need to convert sentences into token IDs, segment IDs (for multi-sentence inputs), and position embeddings that the model understands. We'll cap everything at 128 tokens to avoid memory blowups.

In [ ]:
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f'Tokenizer loaded: {model_name}')
print(f'Vocabulary size: {len(tokenizer)}')
print('Token IDs and segment IDs come from the tokenizer; position embeddings are handled inside BERT.')

In [ ]:
import os

hf_token = os.getenv('HF_TOKEN')

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = None

if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print('Hugging Face login successful.')
    except Exception as err:
        print(f'HF_TOKEN found, but login was skipped: {err}')
else:
    print('No HF_TOKEN found. Using public access (SST-2 is public).')

In [ ]:
print('Loading SST-2 dataset...')
if hf_token:
    dataset = load_dataset('stanfordnlp/sst2', token=hf_token)
else:
    dataset = load_dataset('stanfordnlp/sst2')

max_train_samples = 10000
if len(dataset['train']) > max_train_samples:
    train_dataset = dataset['train'].select(range(max_train_samples))
else:
    train_dataset = dataset['train']

val_dataset = dataset['validation']
test_dataset = dataset['validation']

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'\\nSample from dataset:')
print(f'Text: {train_dataset[0]["sentence"]}')
print(f'Label: {train_dataset[0]["label"]} (0=negative, 1=positive)')

In [ ]:
class SST2Dataset(Dataset):
    \"\"\"\n
    Wrapper around Hugging Face SST-2 data.
    Handles tokenization on-the-fly with fixed sequence length.
    \"\"\"
    def __init__(self, hf_dataset, tokenizer, max_length=128):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        text = item['sentence']
        label = item['label']

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'token_type_ids': encoding['token_type_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


train_data = SST2Dataset(train_dataset, tokenizer, max_length=128)
val_data = SST2Dataset(val_dataset, tokenizer, max_length=128)

print(f'✓ Dataset classes created')
print(f'\\nSample batch:')
sample = train_data[0]
for key, val in sample.items():
    print(f'  {key}: shape {val.shape}, dtype {val.dtype}')

In [ ]:
batch_size = 32
num_workers = 0 if sys.platform.startswith('win') else 2

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

print(f'✓ DataLoaders ready')
print(f'num_workers: {num_workers}')
print(f'Training batches per epoch: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')

---\n\n
## Task 2: Building the Adapter Module\n\n
An Adapter is essentially a small bottleneck network. Given a hidden state from BERT (768 dimensions), we project it down to something tiny (like 64 dims), apply ReLU, then project back up to 768. The whole thing is learnable, but uses way fewer parameters than training the full model.\n\n
The formula looks like:\n
$$\\text{Adapter}(x) = W_2 \\cdot \\text{ReLU}(W_1 \\cdot x + b_1) + b_2$$

In [ ]:
class Adapter(nn.Module):
    \"\"\"\n
    Simple feed-forward adapter for parameter-efficient fine-tuning.\n
    \n
    Architecture:\n
      input (hidden_dim) -> Linear down (bottleneck_dim) -> ReLU ->\n
      -> Linear up (hidden_dim) -> output\n
    \n
    This is placed after self-attention layers in the transformer.\n
    \"\"\"
    def __init__(self, hidden_dim, bottleneck_dim=64):
        super().__init__()
        self.down_proj = nn.Linear(hidden_dim, bottleneck_dim)
        self.activation = nn.ReLU()
        self.up_proj = nn.Linear(bottleneck_dim, hidden_dim)
        
    def forward(self, x):
        x = self.down_proj(x)
        x = self.activation(x)
        x = self.up_proj(x)
        return x


test_adapter = Adapter(hidden_dim=768, bottleneck_dim=64)
test_input = torch.randn(2, 10, 768)
test_output = test_adapter(test_input)

print(f'✓ Adapter module defined')
print(f'Input shape: {test_input.shape}')
print(f'Output shape: {test_output.shape}')

adapter_params = sum(p.numel() for p in test_adapter.parameters())
print(f'\\nAdapter parameters: {adapter_params:,}')
print(f'(vs. BERT\'s ~110M parameters)')

---\n\n## Task 3: Injecting Adapters into BERT\n\nHere's the key move: we load pre-trained BERT, freeze all its weights, and inject an adapter right after the self-attention output inside every transformer block. Only adapter weights and the classification head are trainable, so we keep BERT's core knowledge intact while specializing for sentiment.

In [ ]:
class AdapterEnhancedBertLayer(nn.Module):
    """
    Wraps a pre-trained BertLayer and injects an adapter right after
    the self-attention output, before the feed-forward stage.
    """
    def __init__(self, bert_layer, hidden_dim, bottleneck_dim=64):
        super().__init__()
        self.bert_layer = bert_layer
        self.adapter = Adapter(hidden_dim, bottleneck_dim)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        past_key_value=None,
        output_attentions=False,
    ):
        self_attention_outputs = self.bert_layer.attention(
            hidden_states,
            attention_mask=attention_mask,
            head_mask=head_mask,
            output_attentions=output_attentions,
        )

        attention_output = self_attention_outputs[0]
        other_outputs = self_attention_outputs[1:]

        # Strict PEFT placement: immediately after self-attention.
        attention_output = attention_output + self.adapter(attention_output)

        layer_output = self.bert_layer.feed_forward_chunk(attention_output)
        return (layer_output,) + other_outputs


class BERTWithAdapters(nn.Module):
    """
    BERT model with adapters injected inside each transformer block.

    Pipeline:
    1. Load pre-trained BERT encoder
    2. Freeze base BERT parameters
    3. Inject adapter after self-attention in every layer
    4. Add classification head on top of CLS representation
    5. Train only adapters + classification head
    """
    def __init__(self, model_name, num_labels=2, bottleneck_dim=64):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_dim = self.bert.config.hidden_size

        for param in self.bert.parameters():
            param.requires_grad = False

        self.adapters = nn.ModuleList()
        for i in range(self.bert.config.num_hidden_layers):
            wrapped_layer = AdapterEnhancedBertLayer(
                self.bert.encoder.layer[i],
                hidden_dim=self.hidden_dim,
                bottleneck_dim=bottleneck_dim,
            )
            self.bert.encoder.layer[i] = wrapped_layer
            self.adapters.append(wrapped_layer.adapter)

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hidden_dim, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits


print('Initializing BERT with Adapters...')
model = BERTWithAdapters(model_name='bert-base-uncased', num_labels=2, bottleneck_dim=64)
model = model.to(device)
print(f'✓ Model loaded on {device}')

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


total_params, trainable_params = count_parameters(model)

print('\\n' + '='*70)
print('PARAMETER EFFICIENCY COMPARISON')
print('='*70)
print(f'Total model parameters:       {total_params:,} ({total_params/1e6:.2f}M)')
print(f'Trainable parameters:         {trainable_params:,} ({trainable_params/1e3:.2f}K)')
print(f'Frozen parameters:            {total_params - trainable_params:,} ({(total_params - trainable_params)/1e6:.2f}M)')
print(f'\\nTrainable % of total:         {100*trainable_params/total_params:.3f}%')
print(f'Memory savings:               {100*(1 - trainable_params/total_params):.1f}%')
print('='*70)

In [ ]:
print('\\nGradient configuration check:')

print('\\nBase BERT parameters (should all be False):')
for name, param in model.bert.named_parameters():
    if 'adapter' not in name and param.requires_grad:
        print(f'  WARNING: {name} requires_grad=True')
print('  ✓ All base BERT parameters frozen')

print('\\nAdapters (should all be True):')
for i, adapter in enumerate(model.adapters):
    for name, param in adapter.named_parameters():
        if not param.requires_grad:
            print(f'  WARNING: Adapter {i} {name} requires_grad=False')
print('  ✓ All adapter parameters trainable')

print('\\nClassification head (should all be True):')
for name, param in model.classifier.named_parameters():
    if not param.requires_grad:
        print(f'  WARNING: {name} requires_grad=False')
print('  ✓ Classification head trainable')

---\n\n
## Task 4: Supervised Fine-Tuning & Inference\n\n
Now we train for 3-5 epochs. The frozen BERT weights stay put while adapters learn task-specific patterns. We'll track both training and validation loss to see the model converge.

In [ ]:
num_epochs = 4
learning_rate = 2e-4
optimizer = AdamW(model.parameters(), lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

train_losses = []
val_losses = []
val_accuracies = []

print(f'Training configuration:')
print(f'  Epochs: {num_epochs}')
print(f'  Learning rate: {learning_rate}')
print(f'  Batch size: {batch_size}')
print(f'  Optimizer: AdamW')
print(f'  Loss function: CrossEntropyLoss')

# Quick sanity check: verify every adapter gets gradients.
model.train()
sample_batch = next(iter(train_loader))
input_ids = sample_batch['input_ids'].to(device)
attention_mask = sample_batch['attention_mask'].to(device)
token_type_ids = sample_batch['token_type_ids'].to(device)
labels = sample_batch['label'].to(device)

optimizer.zero_grad()
sample_logits = model(input_ids, attention_mask, token_type_ids)
sample_loss = loss_fn(sample_logits, labels)
sample_loss.backward()

missing_grad_layers = []
for i, adapter in enumerate(model.adapters):
    has_grad = any(param.grad is not None for param in adapter.parameters())
    if not has_grad:
        missing_grad_layers.append(i)

if missing_grad_layers:
    print(f'WARNING: Adapters without gradients: {missing_grad_layers}')
else:
    print('Gradient sanity check passed: all adapters received gradients.')

optimizer.zero_grad()
model.eval()

In [ ]:
def train_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask, token_type_ids)
        loss = loss_fn(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / num_batches
    return avg_loss


def validate(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    num_batches = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for batch in pbar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask, token_type_ids)
            loss = loss_fn(logits, labels)
            
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / num_batches
    accuracy = correct / total
    return avg_loss, accuracy


print('✓ Training and validation functions defined')

In [ ]:
print('\\n' + '='*70)
print('STARTING SUPERVISED FINE-TUNING')
print('='*70 + '\\n')

for epoch in range(num_epochs):
    print(f'\\n--- Epoch {epoch + 1}/{num_epochs} ---')
    
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    train_losses.append(train_loss)
    print(f'Training loss: {train_loss:.4f}')
    
    val_loss, val_accuracy = validate(model, val_loader, loss_fn, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    print(f'Validation loss: {val_loss:.4f}')
    print(f'Validation accuracy: {val_accuracy:.4f}')

print('\\n' + '='*70)
print('TRAINING COMPLETE')
print('='*70)

## Training Results Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)
ax1.plot(epochs_range, train_losses, marker='o', label='Training Loss', linewidth=2, markersize=8)
ax1.plot(epochs_range, val_losses, marker='s', label='Validation Loss', linewidth=2, markersize=8)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training vs Validation Loss Convergence', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epochs_range)

ax2.plot(epochs_range, val_accuracies, marker='D', color='green', linewidth=2, markersize=8)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Validation Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_range)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('training_results.png', dpi=100, bbox_inches='tight')
plt.show()

print('\\n✓ Training plots saved')

---\n\n
## Inference: Testing on Custom Sentences

In [ ]:
def predict_sentiment(text, model, tokenizer, device):
    model.eval()

    encoding = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    token_type_ids = encoding['token_type_ids'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask, token_type_ids)
        probs = F.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred].item()

    label = 'Positive' if pred == 1 else 'Negative'
    return label, confidence


test_sentences = [
    'The movie was absolutely fantastic!',
    'I hated it. Total waste of time.',
    'It was okay, nothing special.',
    'Amazing performance by the lead actor!',
    'Boring and predictable plot.',
    'One of the best films I\'ve ever seen!'
]

print('\\n' + '='*70)
print('SENTIMENT PREDICTION ON CUSTOM SENTENCES')
print('='*70 + '\\n')

for i, sentence in enumerate(test_sentences, 1):
    pred, conf = predict_sentiment(sentence, model, tokenizer, device)
    print(f'{i}. Text: {sentence}')
    print(f'   Prediction: {pred} (confidence: {conf:.4f})')
    print()

user_input_text = 'The movie was absolutely fantastic'
user_pred, user_conf = predict_sentiment(user_input_text, model, tokenizer, device)
print('Single input demo:')
print(f'Input: {user_input_text}')
print(f'Predicted sentiment: {user_pred} (confidence: {user_conf:.4f})')

---\n\n## Analysis: Why PEFT with Adapters?\n\nStandard full fine-tuning of BERT is expensive because every training step updates all ~110M parameters. That increases **compute cost** (full forward/backward over the whole network), **memory cost** (optimizer states + gradients for all weights), and **storage cost** (large checkpoints for each experiment). On limited GPUs, this often means smaller batches, slower training, and higher risk of instability or overfitting.\n\nPEFT with adapters solves this by adding a **set of modular components inside the transformer**. In this notebook, each adapter is injected after self-attention in every encoder block. The base BERT weights stay frozen, so we only train the adapters and the final classifier.\n\nThis design is efficient for three reasons:\n\n1. **Lower memory footprint**: gradients are stored for a small fraction of parameters.\n2. **Faster optimization**: fewer trainable weights means quicker updates.\n3. **Smaller task checkpoints**: adapter-style updates are lightweight compared with full-model snapshots.\n\nFor SST-2, where BERT already has strong language features, adapters let us specialize to sentiment classification without paying the full fine-tuning cost. In practice, this gives a strong accuracy-efficiency tradeoff and is easier to run on standard Colab hardware.

---\n\n
## Summary\n\n
We've built a complete PEFT pipeline:\n\n
1. **Tokenization** — converted SST-2 text into BERT-compatible token sequences with attention masks and segment IDs\n
2. **Adapter design** — created lightweight bottleneck modules (down-project, ReLU, up-project)\n
3. **Architecture integration** — injected adapters into BERT's transformer layers while freezing the base model\n
4. **Training & inference** — fine-tuned adapters on 10K sentiment examples, achieved solid validation accuracy, and built a text prediction pipeline\n\n
The key insight: you don't need to retrain everything to adapt a model. A few strategic, learnable modules can capture task-specific patterns while preserving learned knowledge. This approach scales to production—you can efficiently serve different task-specific adapters with a single frozen BERT core.

In [ ]:
print('\\n' + '='*70)
print('ASSIGNMENT 3 COMPLETE')
print('='*70)
print(f'\\nFinal Metrics:')
print(f'  Best validation accuracy: {max(val_accuracies):.4f}')
print(f'  Final validation loss: {val_losses[-1]:.4f}')
print(f'  Total trainable parameters: {trainable_params:,}')
print(f'  Parameter efficiency: {100*trainable_params/total_params:.3f}%')
print('\\nNote: This solution demonstrates that parameter-efficient fine-tuning')
print('can achieve strong results with minimal computational overhead.')
print('='*70)